In [ ]:
# --- setup -------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/pxr-repo'
!git clone -q https://github.com/pridem755/patient-or-xray.git $REPO 2>/dev/null || (cd $REPO && git pull -q)
%pip install -q -e $REPO

In [ ]:
import importlib
import site
import sys

site.main()
importlib.invalidate_caches()
SRC = f'{REPO}/src'
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pxr
print('pxr loaded from:', pxr.__file__)

In [ ]:
from pathlib import Path

import pandas as pd

from pxr.config import freeze_config, load_config
from pxr.data.splits import (
    assign_folds,
    cell_balance_report,
    fold_balance_report,
    fold_membership,
    plan_validation_allocation,
    validate_folds,
)
from pxr.prereg import render_analysis_plan

cfg = load_config(f'{REPO}/config/study_config.yaml')
ROOT = Path(cfg.paths['drive_root'])
COHORTS = ROOT / cfg.paths['cohorts']
ANALYSIS = ROOT / cfg.paths['analysis']
SPLITS = ROOT / cfg.paths['splits']
SPLITS.mkdir(parents=True, exist_ok=True)

print('config_hash :', cfg.config_hash)
print('training sites :', cfg.training_sites, '(each trains its own model)')
print('descriptive :', cfg.analysis.get('descriptive_sites'))
print('folds :', cfg.n_folds, ' val fraction:', cfg.val_fraction)
print('min val stratum :', cfg.min_val_stratum)
print('stratified on :', cfg.stratify_by)
print('design :', cfg.analysis['design'])

In [ ]:
cohorts = {}
for site in cfg.site_names:
    cohorts[site] = pd.read_parquet(COHORTS / cfg.artifact_name('cohort', site=site))
    print(f'{site:<10} {len(cohorts[site]):>7,} patients')

tiers = pd.read_csv(ANALYSIS / f'label_tiers_{cfg.config_hash}.csv')
coupling = pd.read_csv(ANALYSIS / f'acquisition_coupling_{cfg.config_hash}.csv')
sens_path = ANALYSIS / f'tier_sensitivity_{cfg.config_hash}.csv'
sensitivity = pd.read_csv(sens_path, index_col=0) if sens_path.exists() else None
print('\nprimary family:', tiers.loc[tiers.tier == 'primary', 'label'].tolist())

In [ ]:
# --- assign folds ----------------------------------------------------------------
folds = {}
for site, df in cohorts.items():
    folds[site] = assign_folds(
        df,
        is_training_site=(site in cfg.training_sites),
        stratify_by=cfg.stratify_by,
        n_folds=cfg.n_folds,
        seed=cfg.splits['seed'],
    )
    counts = folds[site]['fold'].value_counts().sort_index().to_dict()
    print(f'{site:<10} {counts}')

# roles for one outer fold, to show the nested structure
site = cfg.training_sites[0]
role = fold_membership(
    0, folds[site],
    stratify_by=cfg.stratify_by,
    n_folds=cfg.n_folds,
    val_fraction=cfg.val_fraction,
    min_val_stratum=cfg.min_val_stratum,
    seed=cfg.splits['seed'],
)
print(f'\n{site} fold 0 roles:', role.value_counts(normalize=True).round(3).to_dict())

In [ ]:
# --- validate folds ----------------------------------------------------------------
report = validate_folds(folds, training_sites=cfg.training_sites, n_folds=cfg.n_folds)
print(report.to_string(index=False))
assert report.ok.all(), 'fold validation failed'
print('\nAll fold invariants hold: one fold per patient, sizes even, external sites clean.')

In [ ]:
# --- check fold invariants ----------------------------------------------------------------
for site in cfg.training_sites:
    tested = pd.Series(0, index=folds[site].index)
    exclusive = True
    for k in range(cfg.n_folds):
        r = fold_membership(k, folds[site], stratify_by=cfg.stratify_by,
                            n_folds=cfg.n_folds, val_fraction=cfg.val_fraction,
                            min_val_stratum=cfg.min_val_stratum, seed=cfg.splits['seed'])
        ids = {role: set(folds[site].loc[r == role, 'patient_id'])
               for role in ('train', 'val', 'test')}
        exclusive &= not (ids['train'] & ids['val'] or ids['train'] & ids['test']
                          or ids['val'] & ids['test'])
        tested += (r == 'test').astype(int)

    one_fold = (folds[site].groupby('patient_id')['fold'].nunique() == 1).all()
    print(f'{site}:')
    print(f'train/val/test mutually exclusive in every fold : {exclusive}')
    print(f'every patient in exactly one outer fold : {bool(one_fold)}')
    print(f'every patient a test patient exactly once : {bool((tested == 1).all())}')
    assert exclusive and one_fold and (tested == 1).all(), f'{site}: leakage guarantee failed'

In [ ]:
# --- plan validation allocation ----------------------------------------------------------------
for site in cfg.training_sites:
    development = folds[site][folds[site]['fold'] != 0]
    plan = plan_validation_allocation(
        development, stratify_by=cfg.stratify_by,
        val_fraction=cfg.val_fraction, min_stratum_size=cfg.min_val_stratum,
    )
    target = round(len(development) * cfg.val_fraction)
    print(f'{site:<10} {len(plan):>3} strata, {int(plan.skipped.sum()):>2} skipped, '
          f'{int(plan.n_val.sum()):>6,} validation patients (target {target:,})')
    if plan.skipped.any():
        print(plan[plan.skipped][['stratum', 'n_patients']].to_string(index=False))

In [ ]:
# --- report fold balance ----------------------------------------------------------------
for site in cfg.training_sites:
    print(f'=== {site} ===')
    print(fold_balance_report(folds[site], cfg.analysis_labels,
                              stratify_by=cfg.stratify_by).to_string(index=False))
    print('  (* = stratified on; unstarred columns were verified, not controlled)\n')

In [ ]:
# --- report thinnest cells ----------------------------------------------------------------
for site in cfg.training_sites:
    print(f'=== {site}: thinnest cells ===')
    cells = cell_balance_report(folds[site], cfg.analysis_labels, strata=cfg.strata)
    print(cells.head(12).to_string(index=False))
    print()

In [ ]:
# --- freeze configuration ----------------------------------------------------------------
frozen = freeze_config(cfg, frozen_dir=f'{REPO}/config/frozen')
print('frozen to:', frozen)

In [ ]:
# --- render analysis plan ----------------------------------------------------------------
cohort_summary = pd.DataFrame([
    {
        'site': site,
        'patients': f'{len(df):,}',
        'AP': f'{(df.view == "AP").mean():.1%}',
        'female': f'{(df.sex == "Female").mean():.1%}',
        'age median': f'{df.age.median():.0f}',
    }
    for site, df in cohorts.items()
])

plan = render_analysis_plan(
    cfg, tiers, coupling, cohort_summary,
    sensitivity=sensitivity,
    frozen_config_path=str(frozen.name),
)

plan_path = Path(REPO) / 'prereg' / 'analysis_plan.md'
plan_path.write_text(plan)
print(f'written: {plan_path}  ({len(plan.splitlines())} lines)\n')
print(plan[:2500])

In [ ]:
# --- save folds and balance reports ----------------------------------------------------------------
all_folds = pd.concat(
    [df[['patient_id', 'site', 'fold']] for df in folds.values()], ignore_index=True
)
out_path = SPLITS / f'folds_{cfg.config_hash}.parquet'
all_folds.to_parquet(out_path, index=False)
for site in cfg.training_sites:
    fold_balance_report(folds[site], cfg.analysis_labels,
                        stratify_by=cfg.stratify_by).to_csv(
        SPLITS / f'fold_balance_{site}_{cfg.config_hash}.csv', index=False)
    cell_balance_report(folds[site], cfg.analysis_labels, strata=cfg.strata).to_csv(
        SPLITS / f'cell_balance_{site}_{cfg.config_hash}.csv', index=False)
print(f'saved {len(all_folds):,} fold assignments to {out_path.name}')

In [ ]:
# --- integrity cell: values for the reproducibility appendix ------------------
print(f'config_hash : {cfg.config_hash}')
print(f'frozen config : {frozen.name}')
print(f'design : {cfg.analysis["design"]}')
print(f'training sites : {cfg.training_sites}')
print(f'folds : {cfg.n_folds}, stratified on {cfg.splits["stratify_by"]}')
for site, df in folds.items():
    counts = df['fold'].value_counts().sort_index().to_dict()
    print(f'{site:<10} {counts}')
print(f'primary family : {tiers.loc[tiers.tier == "primary", "label"].tolist()}')
print(f'exploratory : {tiers.loc[tiers.tier == "exploratory", "label"].tolist()}')
print(f'plan lines : {len(plan.splitlines())}')